In [1]:
import torch
import torch.nn as nn

In [ ]:
class ConvReLU(nn.Module):
    def __init__(self,in_channels,out_channels,kernel_size,padding=0,groups=1):
        super().__init__()
        stride =4 if kernel_size ==11 else 1
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels,out_channels,kernel_size,stride,padding,groups=groups),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.conv(x)

In [ ]:
class ConvReLULRNPool(nn.Module):
    def __init__(self,in_channels,out_channels,kernel_size,padding=0,groups=1):
        super().__init__()
        self.conv = nn.Sequential(
            ConvReLU(in_channels,out_channels,kernel_size,padding,groups=groups),
            nn.LocalResponseNorm(size=5, alpha=0.0005,beta=0.75,k=2),
            nn.MaxPool2d(kernel_size =3, stride=2, padding=0, dilation=1, return_indices=False, ceil_mode=False),
        )

    def forward(self, x):
        return self.conv(x)

In [ ]:
class Classifier(nn.Module):
    def __init__(self,num_classes):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(9216,4096),
            nn.ReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(4096,4096),
            nn.ReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(4096,num_classes),
        )

    def forward(self,x):
        return self.classifier(x)

In [3]:
class AlexNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = ConvReLULRNPool(3,96,11)
        self.conv2 = ConvReLULRNPool(96,256,5,2,2)
        self.conv3 = ConvReLU(256,384,3,1,1)
        self.conv4 = ConvReLU(384,384,3,1,2)
        self.conv5 = ConvReLU(384,256,3,1,2)
        self.classifier = Classifier(1000)
        self.max_pool = nn.MaxPool2d(kernel_size =3, stride=2, padding=0, dilation=1, return_indices=False, ceil_mode=False)

    def forward(self,x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.conv5(x)
        x = self.max_pool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x